![image_1780866172121.png](./image_1780866172121.png "image_1780866172121.png")

In [0]:
# Installing Utilities and Libraries
%pip install databricks-vectorsearch

In [0]:
dbutils.library.restartPython()

In [0]:
# Enable CDC (change data capture) on the final RAG dataset in unity catalog
spark.sql("""
    ALTER TABLE dbx_apps_poc.rag.final_rag_dataset
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)     
""")

In [0]:
# Extend Retention period of the final RAG dataset in Unity catalog 
spark.sql("""

    ALTER TABLE dbx_apps_poc.rag.final_rag_dataset
    SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 30 days')    
""")

In [0]:
# Create a vector index and endpoint
from databricks.vector_search.client import VectorSearchClient

vector_client = VectorSearchClient()

# To recreate the endpoint after deletion: this should excuted only once
vector_client.create_endpoint(
    name="vector_search_endpoint",
    endpoint_type="STANDARD"
)

In [0]:
index = vector_client.create_delta_sync_index(
    endpoint_name="vector_search_endpoint",
    source_table_name="dbx_apps_poc.rag.final_rag_dataset",
    primary_key="id",
    pipeline_type="TRIGGERED",
    embedding_source_column='chunk',
    embedding_model_endpoint_name="databricks-gte-large-en",
    index_name="dbx_apps_poc.rag.rag_vector_index"
)

In [0]:
#Triggering our index - information retrieval: querying the vector index with a sample question
import json
from databricks.vector_search.client import VectorSearchClient

user_question = "can you tell me what hotels are offered by Margies Travel in Dubai?"

result_dict = index.similarity_search(
    query_text=user_question,
    columns=["content_path", "chunk"],
    num_results=10,
    query_type='hybrid'
)

print(result_dict)

In [0]:
for content in result_dict['result']['data_array']:
    print(json.dumps(content, indent=2, ensure_ascii=False))